# The Tokyo Medical University Scandal, as a Bias Case Study

In this notebook I look at the 2018 Tokyo Medical University entrance exam scandal, not as a news story, but as a case study in how a scoring system can produce a large, systematic gap in outcomes between two groups who started out equally capable.

The scandal itself was not caused by a machine learning model, it was people adjusting exam scores by hand. I am covering it here anyway because it keeps showing up in AI ethics discussions, and once I worked through why, the mechanism turned out to be exactly the kind of thing that can also hide inside an automated scoring or ranking system, which is what this notebook is really about.

In this notebook, I will:

- Summarize what was reported about the scandal
- Build a small synthetic simulation of a group-level score adjustment
- Measure the resulting gap with a standard disparate-impact test
- Show a subtler trap: a model can reproduce this same gap even after the protected attribute is removed, if it is trained on the biased outcomes
- Write down what I take from this for any AI system I build or use

This stays small and synthetic on purpose. It is not a re-analysis of the real dataset, which I do not have access to, just a way to make the mechanism concrete enough to reason about.

## 1. Background: What Was Reported

In 2018, Tokyo Medical University admitted that it had been adjusting entrance exam scores for years in a way that reduced the number of women admitted, and that also disadvantaged male applicants who were sitting the exam again after previously failing it, while favoring first-time male applicants.

This came to light during an unrelated investigation into the university, and once it did, several other Japanese medical schools acknowledged similar practices in their own admissions. Reporting at the time described the reasoning given internally as a kind of workforce planning: an assumption that women were more likely to reduce their working hours or leave clinical practice after marriage or childbirth, so admitting fewer of them was framed as easing a doctor shortage.

I am summarizing this from public reporting rather than the university's own data, which was never released in a form I could analyze, so the simulation later in this notebook is illustrative, not a reconstruction of the real numbers.

## 2. Why a Manual Scandal Belongs in an AI Ethics Notebook

No model was involved here, so it would be reasonable to ask why this belongs in a notebook about building AI agents. Three things about it map directly onto problems that show up in automated scoring systems:

- A single adjustment was applied to everyone in a group, rather than any individual being judged on their own record.
- The justification was a prediction about the group's future behavior, not anything about the specific applicant in front of the committee.
- The adjustment was invisible to the people it affected, because the scoring process itself was opaque, so nobody outside the university could check it.

All three of these can happen quietly inside a trained model too, a group-level pattern baked into weights instead of a spreadsheet formula, which is exactly why this case gets used as a teaching example well beyond Japan or medicine.

## 3. A Note on the Numbers Below

Everything from here on is a small synthetic simulation I built myself, using made-up scores from a random number generator, not the university's actual data. I am using it to make the *mechanism* concrete: what happens to outcomes when a blanket, group-level adjustment is applied to a scoring process, even when the two groups start out equally capable by construction.

None of the specific numbers below should be read as a claim about the real scandal's actual figures.

## 4. Simulating Two Equally Able Applicant Groups

I start by generating raw exam scores for two groups, drawn from the exact same distribution. Building them from the same distribution on purpose means any gap I see later in this notebook is coming from something I did to the scores, not from the groups actually differing in ability.

In [ ]:
import random
import statistics

random.seed(42)


def simulate_scores(count, mean=70, spread=10):
    """Returns a list of made-up exam scores drawn from the same distribution."""
    return [round(random.gauss(mean, spread), 1) for _ in range(count)]


group_a_scores = simulate_scores(500)
group_b_scores = simulate_scores(500)

## 5. Checking the Baseline Is Actually Fair

Before I do anything else, I check that the two groups really do start out close to equal. This is the control condition for everything that follows.

In [ ]:
print("Group A mean:", round(statistics.mean(group_a_scores), 2))
print("Group B mean:", round(statistics.mean(group_b_scores), 2))

PASS_MARK = 70


def pass_rate(scores, pass_mark=PASS_MARK):
    """Returns the fraction of scores at or above the pass mark."""
    passed = [s for s in scores if s >= pass_mark]
    return len(passed) / len(scores)


print("Group A pass rate:", round(pass_rate(group_a_scores), 3))
print("Group B pass rate:", round(pass_rate(group_b_scores), 3))

## 6. Applying a Group-Level Score Deduction

Now I apply a single, blanket adjustment to one group's scores, loosely modeled on the mechanism reported in the scandal: a fixed percentage taken off every score in the group, regardless of the individual applicant behind it.

In [ ]:
def apply_deduction(scores, reduction_fraction):
    """Reduces every score in a list by a fixed fraction, as a blanket adjustment."""
    return [round(s * (1 - reduction_fraction), 1) for s in scores]

## 7. Comparing Pass Rates After the Deduction

I apply a modest deduction to group B only and recompute the pass rates. Group A's underlying ability has not changed at all, and neither has group B's, only the scores that got recorded.

In [ ]:
group_b_adjusted = apply_deduction(group_b_scores, reduction_fraction=0.1)

print("Group A pass rate (unchanged):", round(pass_rate(group_a_scores), 3))
print("Group B pass rate (after deduction):", round(pass_rate(group_b_adjusted), 3))

## 8. A Quantitative Way to Flag This: the Four-Fifths Rule

Rather than eyeballing the gap, I want a concrete test. The four-fifths rule is a standard rule of thumb from U.S. employment discrimination law: if one group's selection rate is less than 80% of the highest group's selection rate, the process is flagged for possible adverse impact and needs a closer look. It is not a proof of discrimination on its own, but it is a cheap first check.

In [ ]:
def four_fifths_check(rate_a, rate_b):
    """Returns the selection-rate ratio and whether it fails the four-fifths rule."""
    higher = max(rate_a, rate_b)
    lower = min(rate_a, rate_b)
    ratio = lower / higher if higher > 0 else 1.0
    return ratio, ratio < 0.8

## 9. Testing the Rule on Both Scenarios

I run the check on the baseline (before any deduction) and on the adjusted scenario, to see whether it catches the gap I introduced.

In [ ]:
baseline_ratio, baseline_flag = four_fifths_check(
    pass_rate(group_a_scores), pass_rate(group_b_scores)
)
adjusted_ratio, adjusted_flag = four_fifths_check(
    pass_rate(group_a_scores), pass_rate(group_b_adjusted)
)

print("Baseline ratio:", round(baseline_ratio, 3), "flagged:", baseline_flag)
print("Adjusted ratio:", round(adjusted_ratio, 3), "flagged:", adjusted_flag)

## 10. The Deeper Trap for AI Systems: Biased Labels, Not Just Biased Inputs

A common instinct after seeing this is: just do not give the model the protected attribute, and it cannot discriminate on it. That instinct misses a quieter version of the same problem.

Suppose, instead of a person applying a deduction by hand, someone later trained a system to predict who gets admitted, using the university's own past admission outcomes as the training labels. Even if that system's only input feature is the applicant's raw exam score, as long as it also has some way to tell the two groups apart, a school affiliation, an interview note, anything correlated with the group, it can still fit a stricter effective cutoff for the group whose historical outcomes were suppressed. It never needs to be told which attribute caused that, it just needs the historical labels, which already encode it. This is usually called label bias, and it is harder to catch than a biased input feature, because the training data does not look obviously wrong, it just quietly encodes a past decision that was.

## 11. Simulating Cutoffs Learned From Biased History

To make this concrete, I write a tiny stand-in for a trained model: a per-group score cutoff, fit only to reproduce each group's *historical* admit rate. The cutoff search only ever looks at raw, undeducted scores, the same kind of value a real applicant's exam would produce. What makes the cutoffs differ is the *target* they are each fit to match, group A's target is its real historical pass rate, group B's target is the pass rate produced by the deduction, standing in for a real system that has some correlated signal telling the groups apart.

In [ ]:
def learn_cutoff(scores, target_pass_rate):
    """Finds the raw-score cutoff that reproduces a target pass rate on this data."""
    sorted_scores = sorted(scores, reverse=True)
    index = round(target_pass_rate * len(sorted_scores))
    index = max(0, min(index, len(sorted_scores) - 1))
    return sorted_scores[index]


historical_rate_a = pass_rate(group_a_scores)
historical_rate_b = pass_rate(group_b_adjusted)

cutoff_a = learn_cutoff(group_a_scores, historical_rate_a)
cutoff_b = learn_cutoff(group_b_scores, historical_rate_b)

print("Cutoff learned for group A:", cutoff_a)
print("Cutoff learned for group B:", cutoff_b)

## 12. Testing the Learned Cutoffs on Fresh, Fair Applicants

Now comes the part that matters: I generate a brand-new pool of applicants for both groups, drawn from the same fair distribution as the very first simulation in this notebook, with no deduction applied to anyone, raw ability only. Then I apply the cutoffs that were learned from the biased history above, and check the pass rates.

In [ ]:
new_group_a = simulate_scores(500)
new_group_b = simulate_scores(500)

new_pass_rate_a = pass_rate(new_group_a, pass_mark=cutoff_a)
new_pass_rate_b = pass_rate(new_group_b, pass_mark=cutoff_b)

print("New group A pass rate (fair data, fair cutoff):", round(new_pass_rate_a, 3))
print("New group B pass rate (fair data, biased cutoff):", round(new_pass_rate_b, 3))

ratio, flagged = four_fifths_check(new_pass_rate_a, new_pass_rate_b)
print("Four-fifths ratio:", round(ratio, 3), "flagged:", flagged)

## 13. What This Result Shows

The new applicants in both groups came from the exact same fair distribution, nobody's raw ability was touched this time, and the cutoffs only ever operate on a raw exam score. The gap still shows up, because each cutoff was fit to a history that already contained the deduction, so it carries the deduction forward into scores it has never seen.

That is the version of this problem that is easy to miss when building or adopting an AI system: a pipeline can pass a check for "no protected attributes used as input" and still fail a check on outcomes, because the thing that was biased was never an input feature, it was the target the system was trained to match.